# `zaber_control` Example Notebook

This notebook provides a practical introduction to the `zaber_control` class and demonstrates its main features through a typical workflow.

The examples presented here are intended to help you get started quickly. They are **not** an exhaustive overview of every feature or configuration supported by the class. Instead, they focus on the core functionality and the most common usage patterns, which should cover around **99% of everyday use cases**.

For more advanced applications or specialized configurations, please refer to the class documentation.

`zaber_control` is a Python class to control **Zaber linear motor stages** over a serial connection. It wraps the [Zaber Motion Library](https://software.zaber.com/motion-library/docs/tutorials/install/py) to provide a clean, safe, high-level interface for homing, absolute/relative motion, sinusoidal motion, and built-in oscilloscope data capture.

## Initialization

Parameters of the initialization:

| Parameter | Type | Default | Description |
|---|---|---|---|
| `port` | `str` | — | Serial port name. **Required.** (e.g.: "COM4", "COM5"). If you don't know the port, you can either try every port (not the best idea), or use the [Zaber Launcher](https://www.zaber.com/zaber-launcher) to determine the port.|
| `label` | `str` | `"Zaber Motor"` | Name used in every log message. Useful when controlling multiple stages simultaneously (so not the case in the lab for now). |
| `axis_number` | `int` | `1` | Axis index on the device (1-based). Use `1` for single-axis devices. |
| `auto_home` | `bool` | `False` | If `True`, the stage homes immediately on construction. |
| `auto_close` | `bool` | `True` | If `True`, automatically close the connection when leaving a `with` statement. |
| `default_velocity` | `float` | `1` | Set a default velocity for every functions. If `None`, automatically use the internal default velocity. |
| `default_acceleration` | `float` | `None` | Set a default acceleration for every functions. If `None`, automatically use the internal default acceleration. |
| `wait_until_idle` | `bool` | `True` | If `True`, every motion command blocks until the stage is idle. Set `False` for non-blocking calls. |
| `verb` | `bool` | `True` | If `True`, show every log. |

**Important:** if `auto_home` is `True`, it will perform a homing at the **internal default speed** of the motor, even if you set an default_velocity, which is **6 mm/s**. 

In [1]:
from microwave_cavity import zaber_control

zaber = zaber_control(
    port = "COM4",                  # Usually, it's COM4 or COM5
    label = "Zaber",
    auto_home = False,
    default_velocity = 2,           # mm/s
    verb = False)

Loading BokehJS ...

[Zaber] READY!


**Note:** If the motor is unplugged from its current source, it loses its reference position. Thus, if `auto_home` is `False` because of the semi-high velocity, it's important to move the motor to its zero position. This will give the motor a reference, and then you can use the motor.

In [2]:
zaber.move_absolute(position = 0)

**Usual error:** If you have this type of error

`SerialPortBusyException: SerialPortBusyException: Cannot open serial port: Port is likely already opened by another application. Please close the application that is using the port.`

it means the port is busy. Either you know which python (or jupyter file) was used rigth before, and you can close the connection, or you can disconnect and reconnect the USB connection between the computer and the motor, it walso works.

In [10]:
zaber.close()

[Zaber] Connection closed.


## Motion control

- Positions are in **mm**
- Velocities are in **mm/s**
- Accelerations are in **mm/s²**

The velocity and the acceleration can be provided for each function as supplementary parameters, but it's recommended to define global default velocity and acceleration during the initialization.

There are two types of movements:
- Absolute movement: The motor goes to the absolute position given (the reference is the zero position)
- Relative movement: The motor moves from its current position (used as a local reference)

In [3]:
zaber.move_absolute(
    position = 20,
    velocity = None,            # Use the default velocity
    acceleration = None)        # Use the default acceleration

In [4]:
zaber.move_relative(
    delta_position = -5,
    velocity = None,            # Use the default velocity
    acceleration = None)        # Use the default acceleration

## Parameters of the motor

If you want to know the current position of the motor

In [6]:
pos = zaber.get_position()

print(f"Zaber position: {pos:.6f} mm")

Zaber position: 15.000018 mm


## Other features

More features are available with `zaber_control`. However, they are usually unnecessary for a normal use of the cavity filter. More details are provided in the documentation.

## Close connection

Don't forget to close the connection when you're done with the motor. It helps to avoid `SerialPortBusyException` error for other users (or for yourself)

In [7]:
zaber.close()

[Zaber] Connection closed.
